In [0]:
%pip install openpyxl

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import pandas as pd
import openpyxl
import requests
import re #regex

In [0]:
# Path to download the API
path_munic = "https://ftp.ibge.gov.br/Perfil_Municipios/2024/Base_de_Dados/Base_MUNIC_2024_20251107.xlsx"
munic_file = "Base_MUNIC_2024.xlsx"

In [0]:
# Do the request to the IBGE APIs
file = requests.get(path_munic)

In [0]:
# Returns a exception if occur a error on the request
file.raise_for_status()

In [0]:
with open(munic_file, "wb") as f:
    f.write(file.content)

Workbook (wb) = the complete file

Worksheet (ws) = the individual tabbed pages inside the workbook

In [0]:
# Inspect available sheets
wb = openpyxl.load_workbook(munic_file, read_only=True)
print("Available sheets:", wb.sheetnames)

Available sheets: ['Dicionário', 'Recursos humanos', 'Informática e comunicação', 'Governanca', 'Habitacao', 'Transporte e mobilidade urbana', 'Agropecuária', 'Gestão migratória', 'Igualdade racial', 'Evento climático RS']


In [0]:
type(munic_file)

str

In [0]:
# Take the specific worksheet that we want
info_ws = wb["Informática e comunicação"]

In [0]:
for row in info_ws.iter_rows(max_row=5, values_only=True):
  print(row)

('Cod Munic', 'Uf', 'Cod Uf', 'Desc Mun', 'Populacao', 'Faixa_populacao', 'Regiao', 'Mtic011', 'Mtic012', 'Mtic013', 'Mtic014', 'Mtic015', 'Mtic016', 'Mtic017', 'Mtic018', 'Mtic04', 'Mtic06', 'Mtic071', 'Mtic072', 'Mtic073', 'Mtic074', 'Mtic075', 'Mtic076', 'Mtic077', 'Mtic078', 'Mtic08', 'Mtic091', 'Mtic092', 'Mtic093', 'Mtic10', 'Mtic11', 'Mtic12a1', 'Mtic12a2', 'Mtic12a3', 'Mtic12a4', 'Mtic12a5', 'Mtic12a6', 'Mtic12a7', 'Mtic12a8', 'Mtic12b1', 'Mtic12b2', 'Mtic12b3', 'Mtic12b4', 'Mtic12b5', 'Mtic12b6', 'Mtic12b7', 'Mtic12b8', 'Mtic12b9', 'Mtic12b10', 'Mtic12b11', 'Mtic12b12', 'Mtic12b13', 'Mtic12b14', 'Mtic131', 'Mtic132', 'Mtic133', 'Mtic134', 'Mtic135', 'Mtic136', 'Mtic137', 'Mtic138', 'Mtic139', 'Mtic141', 'Mtic142', 'Mtic143', 'Mtic144', 'Mtic145', 'Mtic146', 'Mtic15', 'Mtic161', 'Mtic162', 'Mtic163', 'Mtic164', 'Mtic165', 'Mtic166', 'Mtic167', 'Mtic171', 'Mtic172', 'Mtic173', 'Mtic174', 'Mtic175', 'Mtic176', 'Mtic181', 'Mtic182', 'Mtic183', 'Mtic184', 'Mtic185', 'Mtic186', 'Mti

## Convert the Worksheets to Dataframe

- "Informatica e Comunicação", Governance and Dicionary worksheets

In [0]:
df_dict = pd.read_excel("Base_MUNIC_2024.xlsx", sheet_name="Dicionário", header=None)

In [0]:
df_info_ws = pd.read_excel(
    "Base_MUNIC_2024.xlsx",
    sheet_name="Informática e comunicação"
)

df_governance_ws = pd.read_excel(
    "Base_MUNIC_2024.xlsx",
    sheet_name="Governanca"
)

In [0]:
df_data = pd.merge(df_info_ws, df_governance_ws, on="Cod Munic", how="inner")

In [0]:
df_dict.head(50)

,0,1,2,3,4,5
0,NaN,NaN,NaN,NaN,NaN,NaN
1,Temas disponíveis,NaN,NaN,NaN,NaN,variável
2,NaN,NaN,NaN,NaN,NaN,NaN
3,1. RECURSOS HUMANOS POR VÍNCULO,NaN,NaN,NaN,NaN,título link
4,2. INFORMÁTICA E COMUNICAÇÃO,NaN,NaN,NaN,NaN,título link
5,3. GOVERNANÇA,NaN,NaN,NaN,NaN,título link
6,4. HABITAÇÃO,NaN,NaN,NaN,NaN,título link
7,5. TRANSPORTE E MOBILIDADE URBANA,NaN,NaN,NaN,NaN,título link
8,6. AGROPECUÁRIA,NaN,NaN,NaN,NaN,título link
9,7. INSTRUMENTOS DE GESTÃO MIGRATÓRIA,NaN,NaN,NaN,NaN,título link


## Extract the Meaning of the Code Variables on the Sheets

- It´s keeped on the df_metabase_info_ws on the final of this section

In [0]:
def obtain_filled_cells(row):
    # Basically catch the values in a row from the dictionary sheet, to retire the columns vacances
    filled_cells = [str(val).strip() for val in row if pd.notna(val) and str(val).strip() != '']

    return filled_cells


def extract_var_code_meaning(row, var_pattern_list):
  """
    Extracts variable codes and their corresponding survey descriptions
    from a raw IBGE MUNIC dictionary row.

    Parameters:
        row (pd.Series): A single row from the unformatted dictionary sheet.
        var_pattern (str): The thematic variable prefix to match (e.g., 'MTIC', 'MGOV', 'MREH').

    Returns:
        dict: {'cod_variable': <str>, 'meaning': <str>} if a code matching the pattern is found,
              otherwise None.
    """

  # It´s used to save the pattern matched on the row using regex
  code_matches = list()

  filled_cells = obtain_filled_cells(row)

  # Verify if the row has values or not
  if not filled_cells:
      return None

  # Create the Regex Expressions
  regex_expr_list = [rf'^{pattern}\d+' for pattern in var_pattern_list]


  # Match if any cell have some pattern codes and add it in code_matches list
  for re_expr in regex_expr_list:
      for val in filled_cells:
        if re.match(re_expr, val, flags=re.IGNORECASE):
           code_matches.append(val)


  # If has any value with some of the code patterns
  if code_matches:
      cod_var = code_matches[0]

      # Find the description or what means of the MTIC code
      desc_cells = [val for val in filled_cells if val != cod_var]

      # If has one string value, get it. If has more, join the them. If doesn´t have, return None meaning to that code
      if desc_cells:
        if len(desc_cells) == 1:
          code_meaning = desc_cells[0]
        else:
          code_meaning = " - ".join(desc_cells) if desc_cells else None

      return {"cod_variable": cod_var, "meaning": code_meaning}

  # Else, returns None
  return None

In [0]:
# Get the meaning of the codes in the "Dicionario" worksheet (Ws)

code_patterns = ['MTIC', 'MGOV']
ws_meta_vars = [res for res in df_dict.apply(extract_var_code_meaning, axis=1, args=(code_patterns,)) if res is not None]

In [0]:
ws_meta_vars

[{'cod_variable': 'MTIC011', 'meaning': 'Correio'},
 {'cod_variable': 'MTIC012', 'meaning': 'Jornais'},
 {'cod_variable': 'MTIC013', 'meaning': 'Websites'},
 {'cod_variable': 'MTIC014', 'meaning': 'WhatsApp'},
 {'cod_variable': 'MTIC015', 'meaning': 'Telefone'},
 {'cod_variable': 'MTIC016', 'meaning': 'Telefone exclusivo'},
 {'cod_variable': 'MTIC017', 'meaning': 'Outros'},
 {'cod_variable': 'MTIC018', 'meaning': 'Não disponibiliza'},
 {'cod_variable': 'MTIC04', 'meaning': 'Tipo de conexão com a internet'},
 {'cod_variable': 'MTIC06',
  'meaning': 'Estrutura organizacional para a área de Tecnologia da Informação e Comunicação (TIC) - existência'},
 {'cod_variable': 'MTIC071',
  'meaning': 'Funcionários efetivos da própria prefeitura'},
 {'cod_variable': 'MTIC072',
  'meaning': 'Servidores cedidos de outro órgão público'},
 {'cod_variable': 'MTIC073', 'meaning': 'Estagiários'},
 {'cod_variable': 'MTIC074',
  'meaning': 'Cargo(s) Comissionado(s) de livre nomeação'},
 {'cod_variable': 'MT

In [0]:
# Transform on Dataframe
df_meta_vars = pd.DataFrame(ws_meta_vars)

In [0]:
df_meta_vars.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 358 entries, 0 to 357
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   cod_variable  358 non-null    object
 1   meaning       358 non-null    object
dtypes: object(2)
memory usage: 5.7+ KB


In [0]:
# Convert the value from the columns to String and keep them clean
df_meta_vars['cod_variable'] = df_meta_vars['cod_variable'].astype(str).str.strip().str.upper()
df_meta_vars['meaning'] = df_meta_vars['meaning'].astype(str).str.strip()

In [0]:
df_meta_vars.iloc[0]

cod_variable    MTIC011
meaning         Correio
Name: 0, dtype: object

In [0]:
df_meta_vars.iloc[357]

cod_variable                                               MGOV36
meaning         Regulamentação ou lei municipal específica sob...
Name: 357, dtype: object

# Pre-Processing the data


In [0]:
df_data.head()

,Cod Munic,Uf_x,Cod Uf_x,Desc Mun_x,Populacao_x,Faixa_populacao_x,Regiao_x,Mtic011,Mtic012,Mtic013,Mtic014,Mtic015,Mtic016,Mtic017,Mtic018,Mtic04,Mtic06,Mtic071,Mtic072,Mtic073,Mtic074,Mtic075,Mtic076,Mtic077,Mtic078,Mtic08,Mtic091,Mtic092,Mtic093,Mtic10,Mtic11,Mtic12a1,Mtic12a2,Mtic12a3,Mtic12a4,Mtic12a5,Mtic12a6,Mtic12a7,Mtic12a8,Mtic12b1,...,Mgov243,Mgov244,Mgov245,Mgov246,Mgov247,Mgov248,Mgov251,Mgov252,Mgov253,Mgov254,Mgov255,Mgov26,Mgov261a,Mgov261b,Mgov261c,Mgov262,Mgov271,Mgov272,Mgov273,Mgov274,Mgov28,Mgov29,Mgov301,Mgov302,Mgov303,Mgov31,Mgov3111,Mgov3112,Mgov3113,Mgov3114,Mgov3115,Mgov3116,Mgov3117,Mgov3118,Mgov3119,Mgov32,Mgov33,Mgov34,Mgov35,Mgov36
0,1100015,RO,11,Alta Floresta DOeste,22853,4 - 20001 até 50000,1 - Norte,Não,Não,Sim,Sim,Sim,Não,Não,Não,Via cabo ou fibra ótica,Não,-,-,-,-,-,-,-,-,Não,Não,Não,Sim,Ativa,WWW.ALTAFLORESTA.RO.GOV.BR,Sim,Não,Sim,Sim,Sim,Sim,Sim,Não,Sim,...,Não,Não,Não,Não,Não,Sim,Não,Não,Não,Não,Sim,Não,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,Não,-,Não,Não,Não
1,1100023,RO,11,Ariquemes,108573,6 - 100001 até 500000,1 - Norte,Não,Não,Sim,Sim,Sim,Não,Não,Não,Via cabo ou fibra ótica,Sim,Sim,Não,Sim,Sim,Não,Não,Não,Não,Sim,Não,Não,Sim,Ativa,WWW.ARIQUEMES.RO.GOV.BR,Sim,Sim,Sim,Sim,Sim,Sim,Sim,Não,Sim,...,Não,Sim,Sim,Não,Não,Sim,Não,Não,Não,Sim,Não,Não,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,Sim,Na Controladoria ou Corregedoria municipal,Sim,Sim,Sim
2,1100031,RO,11,Cabixi,5690,2 - 5001 até 10000,1 - Norte,Não,Não,Sim,Não,Sim,Não,Não,Não,Via cabo ou fibra ótica,Não,-,-,-,-,-,-,-,-,Sim,Não,Não,Sim,Ativa,WWW.CABIXI.RO.GOV.BR,Sim,Sim,Sim,Não,Sim,Sim,Sim,Não,Sim,...,Não,Sim,Não,Sim,Não,Sim,Não,Não,Não,Não,Sim,Não,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,Sim,No gabinete do(a) Prefeito(a),Não,Sim,Sim
3,1100049,RO,11,Cacoal,97637,5 - 50001 até 100000,1 - Norte,Não,Não,Sim,Sim,Sim,Não,Não,Não,Via cabo ou fibra ótica,Sim,Sim,Não,Sim,Sim,Não,Não,Não,Não,Sim,Sim,Não,Não,Ativa,WWW.CACOAL.RO.GOV.BR,Sim,Sim,Sim,Sim,Sim,Sim,Sim,Não,Sim,...,Não,Sim,Sim,Não,Sim,Sim,Não,Não,Não,Não,Sim,Não,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,Sim,No departamento ou área de Tecnologia da Infor...,Não sabe informar,Não,Não
4,1100056,RO,11,Cerejeiras,16975,3 - 10001 até 20000,1 - Norte,Não,Não,Sim,Sim,Não,Não,Não,Não,Via cabo ou fibra ótica,Sim,Sim,Não,Não,Sim,Não,Não,Não,Não,Sim,Não,Não,Sim,Ativa,WWW.CEREJEIRAS.RO.GOV.BR,Sim,Sim,Sim,Sim,Sim,Sim,Sim,Não,Sim,...,Não,Não,Não,Não,Não,Não,Não,Não,Não,Não,Sim,Não,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,Não,-,Não,Não,Não


In [0]:
df_info_ws.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5570 entries, 0 to 5569
Columns: 205 entries, Cod Munic to Mtic387
dtypes: int64(3), object(202)
memory usage: 8.7+ MB


In [0]:
x = 0
for column in df_data.columns:
    print(column, end="\t\t")
    x += 1
    if (x % 7 == 0):
        print("\n")

Cod Munic		Uf_x		Cod Uf_x		Desc Mun_x		Populacao_x		Faixa_populacao_x		Regiao_x		

Mtic011		Mtic012		Mtic013		Mtic014		Mtic015		Mtic016		Mtic017		

Mtic018		Mtic04		Mtic06		Mtic071		Mtic072		Mtic073		Mtic074		

Mtic075		Mtic076		Mtic077		Mtic078		Mtic08		Mtic091		Mtic092		

Mtic093		Mtic10		Mtic11		Mtic12a1		Mtic12a2		Mtic12a3		Mtic12a4		

Mtic12a5		Mtic12a6		Mtic12a7		Mtic12a8		Mtic12b1		Mtic12b2		Mtic12b3		

Mtic12b4		Mtic12b5		Mtic12b6		Mtic12b7		Mtic12b8		Mtic12b9		Mtic12b10		

Mtic12b11		Mtic12b12		Mtic12b13		Mtic12b14		Mtic131		Mtic132		Mtic133		

Mtic134		Mtic135		Mtic136		Mtic137		Mtic138		Mtic139		Mtic141		

Mtic142		Mtic143		Mtic144		Mtic145		Mtic146		Mtic15		Mtic161		

Mtic162		Mtic163		Mtic164		Mtic165		Mtic166		Mtic167		Mtic171		

Mtic172		Mtic173		Mtic174		Mtic175		Mtic176		Mtic181		Mtic182		

Mtic183		Mtic184		Mtic185		Mtic186		Mtic18a1		Mtic18a2		Mtic18a3		

Mtic18a4		Mtic18a5		Mtic18a6		Mtic19		Mtic201		Mtic202		Mtic203		

Mtic204		Mtic205		Mtic206		Mtic211		Mtic212		M